# InterpretDiffusion 재현 실험

이 노트북은 아래 2가지 케이스를 셀 단위로 재현합니다.

1. 코끼리 + 안경(`an elephant` + concept `glasses`)
2. 여성 + 안경(`a woman` + concept `glasses`)

실행 순서:
1) 환경/경로 설정
2) 데이터셋 생성
3) 학습
4) 테스트(단일 프롬프트, Winobias 선택)

> 주의: 학습/생성 모두 GPU 메모리와 시간이 필요합니다.

In [1]:
import os
import json
from pathlib import Path
from typing import List, Dict

import torch
from diffusers import StableDiffusionPipeline

PROJECT_ROOT = Path('/project')
os.chdir(PROJECT_ROOT)

print('cwd:', os.getcwd())
print('cuda available:', torch.cuda.is_available())

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cwd: /project
cuda available: True


## 1) 실험 설정

`a woman with glasses`와 `a woman`의 차이를 `glasses` concept로 학습하고,
학습된 concept를 `an elephant`에 적용해봅니다.

In [2]:
# 실험 공통 설정
DATA_DIR = PROJECT_ROOT / 'datasets' / 'glasses'
EXP_DIR = PROJECT_ROOT / 'exps' / 'exp_glasses'

NUM_SAMPLES = 600          # 빠른 실험: 300~600, 더 안정적이면 1000
NUM_TRAIN_EPOCHS = 10      # 빠른 실험용
NUM_TEST_SAMPLES = 6

CONCEPT_NAME = 'glasses'
BASE_PROMPT = 'a woman'
TARGET_PROMPT = 'a woman with glasses'
APPLY_PROMPTS = ['a woman', 'an elephant']

print('DATA_DIR =', DATA_DIR)
print('EXP_DIR  =', EXP_DIR)

DATA_DIR = /project/datasets/glasses
EXP_DIR  = /project/exps/exp_glasses


## 2) 데이터셋 생성

- 이미지: `a woman with glasses`
- 입력 프롬프트/컨셉 라벨: `a woman` + `glasses`
- 핵심: `concept_dict.json`에 `glasses` 키가 있어야 `KeyError`가 나지 않습니다.

In [3]:
import json
from tqdm.auto import tqdm

DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1) labels/test/concept_dict 저장
labels = [
    [[BASE_PROMPT, [CONCEPT_NAME]]]
    for _ in range(NUM_SAMPLES)
]

test_cfg = [BASE_PROMPT, [CONCEPT_NAME]]
concept_dict = {CONCEPT_NAME: 0}

with open(DATA_DIR / 'labels.json', 'w') as f:
    json.dump(labels, f)
with open(DATA_DIR / 'test.json', 'w') as f:
    json.dump(test_cfg, f)
with open(DATA_DIR / 'concept_dict.json', 'w') as f:
    json.dump(concept_dict, f)

print('metadata saved')

# 2) 학습용 이미지 생성 (TARGET_PROMPT)
pipe = StableDiffusionPipeline.from_pretrained('CompVis/stable-diffusion-v1-4')
pipe = pipe.to('cuda')
pipe.safety_checker = None
pipe.set_progress_bar_config(disable=True)

for i in tqdm(range(NUM_SAMPLES), desc='creating images'):
    image = pipe(TARGET_PROMPT, num_inference_steps=30)[0][0]
    image.save(DATA_DIR / f'{i}.jpg')

print('dataset creation done:', DATA_DIR)

# 메모리 정리
try:
    del pipe
    torch.cuda.empty_cache()
except Exception:
    pass

metadata saved


creating images: 100%|██████████| 600/600 [1:11:20<00:00,  7.13s/it]


dataset creation done: /project/datasets/glasses


## 3) 학습

`train.py`를 서브프로세스로 실행해 `unet.pth`를 만듭니다.
이미 가중치가 있으면 `SKIP_TRAIN_IF_EXISTS=True`로 스킵할 수 있습니다.

In [ ]:
import subprocess

SKIP_TRAIN_IF_EXISTS = True
weight_path = EXP_DIR / 'unet.pth'

if SKIP_TRAIN_IF_EXISTS and weight_path.exists():
    print('skip training, found:', weight_path)
else:
    EXP_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', 'train.py',
        '--train_data_dir', str(DATA_DIR),
        '--output_dir', str(EXP_DIR),
        '--num_train_epochs', str(NUM_TRAIN_EPOCHS),
        '--log_every_steps', '500',
        '--log_every_epochs', '2',
    ]
    print('running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('trained weight:', weight_path, 'exists=', weight_path.exists())

## 4) 테스트

같은 concept(`glasses`)를 `a woman`, `an elephant`에 각각 주입해 생성합니다.

In [ ]:
import subprocess

for p in APPLY_PROMPTS:
    out_dir_name = p.replace(' ', '_')
    cmd = [
        'python', 'test.py',
        '--train_data_dir', str(DATA_DIR),
        '--output_dir', str(EXP_DIR),
        '--evaluation_type', 'eval',
        '--num_test_samples', str(NUM_TEST_SAMPLES),
        '--prompt', p,
        '--concept', CONCEPT_NAME,
        '--image_dir', f'images_{out_dir_name}',
    ]
    print('running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('done')
print('woman images   ->', EXP_DIR / 'images_a_woman')
print('elephant images->', EXP_DIR / 'images_an_elephant')

In [ ]:
# 결과 미리보기
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt


def preview_folder(folder: Path, title: str, max_n: int = 6):
    imgs = sorted(folder.glob('*.jpg'))[:max_n]
    if not imgs:
        print('no images in', folder)
        return

    fig, axes = plt.subplots(1, len(imgs), figsize=(3*len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]

    for ax, p in zip(axes, imgs):
        ax.imshow(Image.open(p))
        ax.set_title(p.name)
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

preview_folder(EXP_DIR / 'images_a_woman', 'a woman + glasses concept')
preview_folder(EXP_DIR / 'images_an_elephant', 'an elephant + glasses concept')